### 🧩 Step 1 — Load the Cleaned Dataset
We begin by loading the cleaned FPL dataset that already includes the `form` column.
This will be our starting point before we engineer the target column and finalize features.


In [20]:
import pandas as pd

# Load cleaned dataset (make sure the filename matches your actual file)
df = pd.read_csv("../data/cleaned/cleaned_merged_seasons_with_form.csv")

# Display basic info
df.info()
df.head()


ERROR! Session/line number was not unique in database. History logging moved to new session 10
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 96169 entries, 0 to 96168
Data columns (total 38 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   season_x           96169 non-null  object 
 1   name               96169 non-null  object 
 2   position           96169 non-null  object 
 3   team_x             96169 non-null  object 
 4   assists            96169 non-null  int64  
 5   bonus              96169 non-null  int64  
 6   bps                96169 non-null  int64  
 7   clean_sheets       96169 non-null  int64  
 8   creativity         96169 non-null  float64
 9   element            96169 non-null  int64  
 10  fixture            96169 non-null  int64  
 11  goals_conceded     96169 non-null  int64  
 12  goals_scored       96169 non-null  int64  
 13  ict_index          96169 non-null  float64
 14  influence          9616

,season_x,name,position,team_x,assists,bonus,bps,clean_sheets,creativity,element,...,threat,total_points,transfers_balance,transfers_in,transfers_out,value,was_home,yellow_cards,GW,form
0,2020-21,Aaron Connolly,FWD,Brighton,0,0,-3,0,0.3,78,...,32.0,1,0,0,0,55,True,0,1,0.100000
1,2020-21,Aaron Connolly,FWD,Brighton,0,2,27,1,11.3,78,...,23.0,8,-1161,5332,6493,55,False,0,2,0.450000
2,2020-21,Aaron Connolly,FWD,Brighton,0,0,2,0,12.1,78,...,8.0,2,13526,26823,13297,55,True,0,3,0.366667
3,2020-21,Aaron Connolly,FWD,Brighton,0,0,7,0,0.3,78,...,4.0,2,-1311,10399,11710,55,False,0,4,0.325000
4,2020-21,Aaron Connolly,FWD,Brighton,1,0,13,0,10.3,78,...,2.0,4,-8992,5860,14852,55,False,0,5,0.400000


### 🧮 Step 2 — Sort by Player and Gameweek
To ensure that the shifting operation for the target column is done correctly,  
we sort the dataset by each player's name and the chronological order of gameweeks.


In [21]:
df = df.sort_values(["name", "GW"]).reset_index(drop=True)


### 🎯 Step 3 — Create the Target Column (`upcoming_total_points`)
We want to predict each player's total points for the *following* gameweek.  
So for each player, we shift the `total_points` column one step upward.  
The last gameweek for each player will have no "next week" value, so those rows are dropped.


In [22]:
df["upcoming_total_points"] = df.groupby("name")["total_points"].shift(-1)
df = df.dropna(subset=["upcoming_total_points"]).reset_index(drop=True)

df[["name", "GW", "total_points", "upcoming_total_points"]].head(10)



,name,GW,total_points,upcoming_total_points
0,Aaron Connolly,1,1,0.0
1,Aaron Connolly,1,0,8.0
2,Aaron Connolly,2,8,1.0
3,Aaron Connolly,2,1,2.0
4,Aaron Connolly,3,2,0.0
5,Aaron Connolly,3,0,2.0
6,Aaron Connolly,4,2,0.0
7,Aaron Connolly,4,0,4.0
8,Aaron Connolly,5,4,0.0
9,Aaron Connolly,5,0,0.0


### 🧹 Step 4 — Remove Irrelevant Columns
According to the project description, we exclude columns that depend on player popularity
or are identifiers that don’t contribute to predictive modeling.


In [23]:
drop_cols = [
    "transfers_in", "transfers_out", "transfers_balance",
    "element", "season_x"
]

df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors="ignore")


### ⚙️ Step 5 — Select Match-Related and Player-Related Features
We now focus on columns describing performance or player characteristics.  
These will serve as our input features for the regression model.


In [24]:
feature_cols = [
    "total_points",  
    "minutes", "goals_scored", "assists", "bonus", "bps",
    "clean_sheets", "creativity", "influence", "threat",
    "ict_index", "form", "value", "was_home", "yellow_cards", "red_cards",
    "position", "team_x"
]

target_col = "upcoming_total_points"

keep_cols = ["name", "GW"] + feature_cols + [target_col]
df = df[[c for c in keep_cols if c in df.columns]]


### 🔠 Step 6 — Encode Categorical Variables
We convert text features like `team` and `position` into numeric dummy variables.
`drop_first=True` avoids multicollinearity by omitting one category from each.


In [25]:
df["was_home"] = df["was_home"].astype(int)

df = pd.get_dummies(df, columns=["position", "team_x"], drop_first=True)
print("Categorical features encoded successfully.")


Categorical features encoded successfully.


### 💾 Step 7 — Save Final Prepared Dataset
The dataset is now fully prepared for model training.
We save it as `final_prepared_dataset.csv`, ready for splitting into train/test sets.


In [ ]:
output_path = "../data/cleaned/final_prepared_dataset.csv"
df.to_csv(output_path, index=False)

print(f"✅ Final dataset saved as '{output_path}'")
print("Shape:", df.shape)


✅ Final dataset saved as '../data/cleaned/final_prepared_dataset_correct.csv'
Shape: (42654, 46)


### 📊  Sanity Check
Verify that there are no missing values and inspect summary statistics before splitting.


In [27]:
print(df.isna().sum().sum(), "missing values in the final dataset.")
df.describe().T.head(15)


0 missing values in the final dataset.


,count,mean,std,min,25%,50%,75%,max
GW,94842.0,20.531052,10.825246,1.0,11.0,21.00,30.000,38.0
total_points,94842.0,1.386601,2.547781,-7.0,0.0,0.00,2.000,29.0
minutes,94842.0,32.627654,40.620976,0.0,0.0,0.00,90.000,90.0
goals_scored,94842.0,0.045644,0.233916,0.0,0.0,0.00,0.000,4.0
assists,94842.0,0.041258,0.215967,0.0,0.0,0.00,0.000,4.0
bonus,94842.0,0.109340,0.493675,0.0,0.0,0.00,0.000,3.0
bps,94842.0,6.111628,9.867947,-21.0,0.0,0.00,10.000,128.0
clean_sheets,94842.0,0.107695,0.309996,0.0,0.0,0.00,0.000,1.0
creativity,94842.0,4.796833,10.747754,0.0,0.0,0.00,2.600,170.9
influence,94842.0,7.240136,12.930294,0.0,0.0,0.00,10.800,163.6


In [33]:
df.shape
df.columns


Index(['name', 'GW', 'total_points', 'minutes', 'goals_scored', 'assists',
       'bonus', 'bps', 'clean_sheets', 'creativity', 'influence', 'threat',
       'ict_index', 'form', 'value', 'was_home', 'yellow_cards', 'red_cards',
       'upcoming_total_points', 'position_FWD', 'position_GK', 'position_MID',
       'team_x_Aston Villa', 'team_x_Bournemouth', 'team_x_Brentford',
       'team_x_Brighton', 'team_x_Burnley', 'team_x_Chelsea',
       'team_x_Crystal Palace', 'team_x_Everton', 'team_x_Fulham',
       'team_x_Leeds', 'team_x_Leicester', 'team_x_Liverpool',
       'team_x_Man City', 'team_x_Man Utd', 'team_x_Newcastle',
       'team_x_Norwich', 'team_x_Nott'm Forest', 'team_x_Sheffield Utd',
       'team_x_Southampton', 'team_x_Spurs', 'team_x_Watford',
       'team_x_West Brom', 'team_x_West Ham', 'team_x_Wolves'],
      dtype='object')

In [29]:
sample = df[df["name"] == "Erling Haaland"][["GW", "total_points", "upcoming_total_points"]].head(10)
print(sample)


       GW  total_points  upcoming_total_points
27468   1            13                    5.0
27469   2             5                    6.0
27470   3             6                   17.0
27471   4            17                   17.0
27472   5            17                    9.0
27473   6             9                    6.0
27474   8             6                   23.0
27475   9            23                    6.0
27476  10             6                    2.0
27477  11             2                   13.0


In [30]:
# Remove duplicate (player, GW) entries by keeping the one where the player actually played
df = df.sort_values(["name", "GW", "minutes"], ascending=[True, True, False])

# Drop duplicates — keep the one with highest minutes for each player/week
df = df.drop_duplicates(subset=["name", "GW"], keep="first").reset_index(drop=True)


In [31]:
dupes = df.duplicated(subset=["name", "GW"]).sum()
print("Duplicate (name, GW) pairs:", dupes)


Duplicate (name, GW) pairs: 0
